# ECG Arrhythmia Classification with a CNN-BiLSTM-Attention Network

This notebook trains a deep learning model that classifies single ECG heartbeats into the five
AAMI EC57 classes, using the MIT-BIH Arrhythmia Database in its pre-segmented form (Kaggle *heartbeat*
dataset, 187 samples per beat at 125 Hz).

| Class | Code | Description |
|---|---|---|
| 0 | N | Normal beat (incl. bundle branch block, nodal escape) |
| 1 | S | Supraventricular ectopic beat |
| 2 | V | Ventricular ectopic beat |
| 3 | F | Fusion of ventricular and normal beat |
| 4 | Q | Unknown / paced beat |

**Pipeline overview**

1. Load the training and test splits and hold out 10% of the training data for validation.
2. Standardise every time step (`StandardScaler`, fitted on the training split only).
3. Apply random augmentation on the GPU in every batch (time stretch, amplitude scaling, baseline wander, noise).
4. Train `ECG_CNN_LSTM_Deep` (4 convolutional blocks → 2-layer BiLSTM → attention pooling → fully connected head).
5. Select the checkpoint by validation accuracy and evaluate once on the untouched test set.
6. Optionally train a second architecture (1-D ResNet with squeeze-and-excitation) and compare ensembles.

**Training design choices**

| Choice | Reason |
|---|---|
| Fresh augmentation in every epoch | The model never sees the exact same beat twice, which reduces memorisation |
| Mild oversampling of the S and F classes only | Corrects class imbalance without causing many false alarms on rare classes |
| Label smoothing + exponential moving average (EMA) of weights | Better-calibrated, more stable predictions |
| `OneCycleLR`, AdamW, mixed precision | Fast and stable convergence on a single GPU |
| Checkpoint chosen on the validation split | The test set is never used for model selection |

**Note on accuracy.** The MIT-BIH beat labels include genuinely ambiguous beats (especially in the
Supraventricular and Fusion classes), so a perfect test score is not realistic; a score of 100% would
point to leakage between the training and test data rather than a better model.

**Running on Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
The trained files are written to `models/best_model.pth` and `models/scaler.pkl`.

## 1. Setup

In [ ]:
import os
import time
import math
import random
import zipfile
import warnings
warnings.filterwarnings('ignore')

# Data processing
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.swa_utils import AveragedModel

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'

CONFIG = {
    'epochs': 60,
    'batch_size': 256,
    'lr': 2e-3,                     # OneCycle peak learning rate
    'weight_decay': 1e-4,
    'label_smoothing': 0.05,
    'ema_decay': 0.999,
    'dropout': 0.3,                 # training only; dropout is inactive at inference time
    'val_size': 0.10,
    'oversample': {1: 3, 3: 6},     # Supraventricular x3, Fusion x6 (fresh augmentation each time)
    'aug_prob': 0.5,
    'run_ensemble': True,           # Section 8: optional ResNet-SE model + ensemble comparison
}

class_names = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unknown']
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

print("All libraries imported")
print(f"PyTorch: {torch.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if use_amp else " (no GPU found: training will be slow)"))
if sklearn.__version__ != '1.6.1':
    print("Note: the web app's backend pins scikit-learn 1.6.1. A different version here only causes a "
          "warning when scaler.pkl is loaded; run `!pip install -q scikit-learn==1.6.1` and restart to match.")

## 2. Load the MIT-BIH heartbeat dataset
The dataset is used in the pre-segmented form published on Kaggle (`shayanfazeli/heartbeat`), derived from
the PhysioNet MIT-BIH Arrhythmia Database. Each row holds one heartbeat (187 values, zero-padded) followed by
its class label.

The cell uses `mitbih_train.csv` / `mitbih_test.csv` if they are present, otherwise extracts an uploaded
`archive.zip`, and otherwise downloads the dataset with `kagglehub`.

In [ ]:
DATA_FILES = ('mitbih_train.csv', 'mitbih_test.csv')

def find_data_dir():
    for d in ['.', 'archive', '/content', '/content/archive']:
        if all(os.path.exists(os.path.join(d, f)) for f in DATA_FILES):
            return d
    return None

data_dir = find_data_dir()
if data_dir is None and os.path.exists('archive.zip'):
    print("Extracting archive.zip...")
    zipfile.ZipFile('archive.zip').extractall('.')
    data_dir = find_data_dir()
if data_dir is None:
    print("Downloading from Kaggle...")
    import kagglehub
    data_dir = kagglehub.dataset_download('shayanfazeli/heartbeat')
print(f"Data directory: {data_dir}")

train_df = pd.read_csv(os.path.join(data_dir, 'mitbih_train.csv'), header=None).dropna()
test_df = pd.read_csv(os.path.join(data_dir, 'mitbih_test.csv'), header=None).dropna()
print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")

X_full = train_df.iloc[:, :-1].values.astype(np.float32)
y_full = train_df.iloc[:, -1].values.astype(int)
X_test_raw = test_df.iloc[:, :-1].values.astype(np.float32)
y_test = test_df.iloc[:, -1].values.astype(int)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_full, y_full, test_size=CONFIG['val_size'], stratify=y_full, random_state=SEED
)
print(f"\nTrain: {len(X_train_raw):,} | Val: {len(X_val_raw):,} | Test: {len(X_test_raw):,}")

print("\nClass distribution (train split):")
for i, name in enumerate(class_names):
    print(f"  {name:<17} {(y_train == i).sum():>7,}")

## 3. Normalisation
A `StandardScaler` standardises each of the 187 time steps. It is fitted on the training split only and
saved to `models/scaler.pkl`, so inference applies exactly the same transform.

Training batches are kept **raw** on the GPU, so augmentation acts on the real signal; they are standardised
with the same scaler statistics afterwards. Validation and test data are transformed once with
`scaler.transform`, exactly as at inference time.

In [ ]:
scaler = StandardScaler().fit(X_train_raw)
joblib.dump(scaler, 'models/scaler.pkl')
print("Scaler saved to models/scaler.pkl")

mean_t = torch.tensor(scaler.mean_, dtype=torch.float32, device=device).view(1, 1, -1)
scale_t = torch.tensor(scaler.scale_, dtype=torch.float32, device=device).view(1, 1, -1)

def normalize(x):
    return (x - mean_t) / scale_t

def to_device(X):
    return torch.tensor(X, dtype=torch.float32, device=device).unsqueeze(1)  # (N, 1, 187)

X_train_raw_t = to_device(X_train_raw)
y_train_t = torch.tensor(y_train, dtype=torch.long, device=device)
X_val_raw_t = to_device(X_val_raw)
X_test_raw_t = to_device(X_test_raw)

# Normalised evaluation sets: same transform as used at inference time
X_val_t = to_device(scaler.transform(X_val_raw))
X_test_t = to_device(scaler.transform(X_test_raw))
print(f"Tensors on {device}: train {tuple(X_train_raw_t.shape)}, val {tuple(X_val_t.shape)}, test {tuple(X_test_t.shape)}")

## 4. On-the-fly GPU augmentation
Augmentation is applied per sample in every batch, with new random parameters each time a beat is seen:

* **Time stretch** (0.9–1.1×): simulates small heart-rate differences
* **Amplitude scaling** (0.9–1.1×): simulates differences in electrode contact and gain
* **Baseline wander** (slow sine wave): simulates respiration and movement
* **Gaussian noise**: simulates muscle and electrical noise

The additive transforms only affect the beat itself, not the zero-padded tail of each row.

In [ ]:
def time_stretch(x, alpha):
    # x: (B, 1, L), alpha: (B, 1). alpha > 1 compresses the beat, < 1 stretches it; tail is zero-padded.
    B, _, L = x.shape
    pos = torch.arange(L, device=x.device, dtype=torch.float32).unsqueeze(0) * alpha
    i0 = pos.floor().long().clamp(0, L - 1)
    i1 = (i0 + 1).clamp(max=L - 1)
    w = (pos - i0.float()).clamp(0, 1)
    s = x.squeeze(1)
    out = s.gather(1, i0) * (1 - w) + s.gather(1, i1) * w
    out = torch.where(pos <= L - 1, out, torch.zeros_like(out))
    return out.unsqueeze(1)

def augment_batch(x, p=CONFIG['aug_prob']):
    B, _, L = x.shape
    dev = x.device

    def coin():
        return (torch.rand(B, 1, 1, device=dev) < p).float()

    # 1. Time stretch 0.9–1.1
    alpha = torch.empty(B, 1, device=dev).uniform_(0.9, 1.1)
    alpha = torch.where(torch.rand(B, 1, device=dev) < p, alpha, torch.ones_like(alpha))
    x = time_stretch(x, alpha)

    # Mask of the real beat (everything up to the last non-zero sample)
    valid = (x != 0).float().flip(-1).cummax(-1).values.flip(-1)

    # 2. Amplitude scaling 0.9–1.1
    amp = torch.empty(B, 1, 1, device=dev).uniform_(0.9, 1.1)
    x = x * (1 + coin() * (amp - 1))

    # 3. Baseline wander (slow sine)
    t = torch.linspace(0, 1, L, device=dev).view(1, 1, L)
    freq = torch.empty(B, 1, 1, device=dev).uniform_(0.1, 0.6)
    phase = torch.empty(B, 1, 1, device=dev).uniform_(0, 2 * math.pi)
    wander_amp = torch.empty(B, 1, 1, device=dev).uniform_(0, 0.05)
    x = x + coin() * valid * wander_amp * torch.sin(2 * math.pi * freq * t + phase)

    # 4. Gaussian noise
    sigma = torch.empty(B, 1, 1, device=dev).uniform_(0, 0.02)
    x = x + coin() * valid * sigma * torch.randn_like(x)
    return x

# Visual sanity check: original vs 4 augmented variants of one beat per class
fig, axes = plt.subplots(1, 5, figsize=(22, 3.5))
for c, ax in enumerate(axes):
    beat = X_train_raw_t[(y_train_t == c).nonzero()[0, 0]].unsqueeze(0)
    for _ in range(4):
        ax.plot(augment_batch(beat, p=1.0)[0, 0].cpu(), alpha=0.5, linewidth=1)
    ax.plot(beat[0, 0].cpu(), color='black', linewidth=2, label='original')
    ax.set_title(class_names[c], fontweight='bold')
    ax.set_xlim(0, 187)
axes[0].legend()
plt.suptitle('On-the-fly augmentation examples', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Models
**`ECG_CNN_LSTM_Deep`** (main model, used by the web application in `backend/model_definition.py`):

* 4 × (Conv1d → BatchNorm → ReLU), with max-pooling after the first three blocks (187 → 23 time steps)
* 2-layer bidirectional LSTM (192 units per direction) over the convolutional features
* Attention pooling: a learned weight per time step, used to form a weighted sum of the LSTM outputs
* 4 fully connected layers with BatchNorm and dropout → 5 class scores

**`ECG_ResNet_SE`** (optional second model for the ensemble in Section 8): a 1-D residual network with
squeeze-and-excitation channel attention, followed by combined average and max pooling.

In [ ]:
# CNN-LSTM model with attention (same definition as backend/model_definition.py)
class ECG_CNN_LSTM_Deep(nn.Module):
    def __init__(self, num_classes=5, dropout=0.45):
        super(ECG_CNN_LSTM_Deep, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool1 = nn.MaxPool1d(2, 2)

        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.pool2 = nn.MaxPool1d(2, 2)

        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(256)
        self.pool3 = nn.MaxPool1d(2, 2)

        self.conv4 = nn.Conv1d(256, 512, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm1d(512)

        # LSTM layers
        self.lstm = nn.LSTM(input_size=512, hidden_size=192, num_layers=2,
                            batch_first=True, dropout=dropout, bidirectional=True)

        # Attention mechanism
        self.attention = nn.Linear(384, 1)

        # Fully connected layers
        self.dropout1 = nn.Dropout(dropout)
        self.fc1 = nn.Linear(384, 256)
        self.bn_fc1 = nn.BatchNorm1d(256)

        self.dropout2 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(256, 128)
        self.bn_fc2 = nn.BatchNorm1d(128)

        self.dropout3 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(128, 64)
        self.bn_fc3 = nn.BatchNorm1d(64)

        self.dropout4 = nn.Dropout(dropout)
        self.fc4 = nn.Linear(64, num_classes)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool3(x)
        x = self.relu(self.bn4(self.conv4(x)))
        x = x.transpose(1, 2)  # (batch, length, features)
        lstm_out, _ = self.lstm(x)
        attention_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attention_weights * lstm_out, dim=1)
        x = self.dropout1(context)
        x = self.relu(self.bn_fc1(self.fc1(x)))
        x = self.dropout2(x)
        x = self.relu(self.bn_fc2(self.fc2(x)))
        x = self.dropout3(x)
        x = self.relu(self.bn_fc3(self.fc3(x)))
        x = self.dropout4(x)
        x = self.fc4(x)
        return x


# Optional second model: 1D ResNet with Squeeze-and-Excitation (ensemble only)
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(channels, channels // reduction), nn.ReLU(),
                                nn.Linear(channels // reduction, channels), nn.Sigmoid())

    def forward(self, x):
        return x * self.fc(x.mean(-1)).unsqueeze(-1)

class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, kernel_size=5):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, stride=stride, padding=pad, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.se = SEBlock(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
                                          nn.BatchNorm1d(out_ch))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.se(self.bn2(self.conv2(out)))
        return F.relu(out + self.shortcut(x))

class ECG_ResNet_SE(nn.Module):
    def __init__(self, num_classes=5, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(1, 64, 7, padding=3, bias=False), nn.BatchNorm1d(64), nn.ReLU())
        self.layers = nn.Sequential(
            ResBlock1D(64, 64), ResBlock1D(64, 64),
            ResBlock1D(64, 128, stride=2), ResBlock1D(128, 128),
            ResBlock1D(128, 256, stride=2), ResBlock1D(256, 256),
            ResBlock1D(256, 256, stride=2, kernel_size=3),
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.layers(self.stem(x))
        x = torch.cat([x.mean(-1), x.amax(-1)], dim=1)
        return self.fc(self.dropout(x))

for m in [ECG_CNN_LSTM_Deep(), ECG_ResNet_SE()]:
    print(f"{m.__class__.__name__}: {sum(p.numel() for p in m.parameters()):,} parameters")

## 6. Training
Training uses AdamW, a OneCycle learning-rate schedule, label smoothing, gradient clipping and mixed precision.
An exponential moving average (EMA) of the weights is evaluated on the validation split after every epoch, and
the best EMA weights (by accuracy, ties broken by macro-F1) are saved to `models/best_model.pth`.

In [ ]:
def build_train_index(y_np, oversample):
    idx = [np.arange(len(y_np))]
    for c, k in oversample.items():
        idx.append(np.tile(np.where(y_np == c)[0], k - 1))
    return torch.tensor(np.concatenate(idx), device=device)

train_index = build_train_index(y_train, CONFIG['oversample'])
print(f"Samples per epoch (with oversampling): {len(train_index):,}")
for i, name in enumerate(class_names):
    print(f"  {name:<17} {(y_train_t[train_index] == i).sum().item():>7,}")

@torch.no_grad()
def ema_with_warmup(ema_params, current_params, num_averaged):
    # EMA whose decay ramps up from ~0.1 to CONFIG['ema_decay'] so early epochs aren't stuck near the random init
    n = float(num_averaged)
    decay = min(CONFIG['ema_decay'], (1 + n) / (10 + n))
    if torch.is_floating_point(ema_params[0]):
        torch._foreach_lerp_(ema_params, current_params, 1 - decay)
    else:
        for p_ema, p_model in zip(ema_params, current_params):
            p_ema.copy_(p_model)

@torch.no_grad()
def predict_proba(model, X, batch_size=2048):
    model.eval()
    out = []
    for i in range(0, len(X), batch_size):
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(X[i:i + batch_size])
        out.append(torch.softmax(logits.float(), dim=1))
    return torch.cat(out).cpu().numpy()

def train_model(model, save_path, seed=SEED):
    seed_everything(seed)
    model = model.to(device)
    bs, epochs = CONFIG['batch_size'], CONFIG['epochs']
    steps_per_epoch = len(train_index) // bs  # drop last partial batch (BatchNorm needs > 1 sample)

    criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=CONFIG['lr'], epochs=epochs, steps_per_epoch=steps_per_epoch, pct_start=0.15
    )
    ema = AveragedModel(model, multi_avg_fn=ema_with_warmup, use_buffers=True)
    grad_scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    history = {'train_loss': [], 'val_acc': [], 'val_f1': []}
    best = (-1.0, -1.0)
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        perm = train_index[torch.randperm(len(train_index), device=device)]
        total_loss = 0.0
        for step in range(steps_per_epoch):
            b = perm[step * bs:(step + 1) * bs]
            xb = normalize(augment_batch(X_train_raw_t[b]))
            yb = y_train_t[b]

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = criterion(model(xb), yb)
            optimizer.zero_grad(set_to_none=True)
            grad_scaler.scale(loss).backward()
            grad_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            grad_scaler.step(optimizer)
            grad_scaler.update()
            scheduler.step()
            ema.update_parameters(model)
            total_loss += loss.item()

        val_pred = predict_proba(ema.module, X_val_t).argmax(1)
        val_acc = accuracy_score(y_val, val_pred)
        val_f1 = f1_score(y_val, val_pred, average='macro')
        history['train_loss'].append(total_loss / steps_per_epoch)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        marker = ''
        if (val_acc, val_f1) > best:
            best = (val_acc, val_f1)
            torch.save(ema.module.state_dict(), save_path)
            marker = '  (saved)'
        print(f"Epoch [{epoch + 1:2d}/{epochs}] | Loss: {history['train_loss'][-1]:.4f} | "
              f"Val Acc: {val_acc:.4f} F1: {val_f1:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}{marker}")

    print(f"\nTraining completed in {(time.time() - start_time) / 60:.1f} minutes")
    print(f"Best validation accuracy: {best[0]:.4f} (macro-F1 {best[1]:.4f}) → {save_path}")
    model.load_state_dict(torch.load(save_path, map_location=device))
    return model, history

print("=" * 80)
print("TRAINING ECG_CNN_LSTM_Deep (main model)")
print("=" * 80)
model_main, history_main = train_model(ECG_CNN_LSTM_Deep(num_classes=5, dropout=CONFIG['dropout']),
                                       'models/best_model.pth')

## 7. Test-set evaluation
The selected checkpoint is evaluated once on the held-out test split. Because the classes are highly imbalanced
(most beats are Normal), the macro-averaged F1 score and the per-class recall are more informative than overall
accuracy.

In [ ]:
probs_main = predict_proba(model_main, X_test_t)
pred_main = probs_main.argmax(1)
test_acc = accuracy_score(y_test, pred_main)

print("=" * 80)
print("FINAL EVALUATION ON TEST SET")
print("=" * 80)
print(f"  Accuracy:      {test_acc:.4f}")
print(f"  F1 (macro):    {f1_score(y_test, pred_main, average='macro'):.4f}")
print(f"  F1 (weighted): {f1_score(y_test, pred_main, average='weighted'):.4f}")
print("\nPer-class classification report:")
print(classification_report(y_test, pred_main, target_names=class_names, digits=4))

summary = pd.DataFrame([{
    'Model': 'ECG_CNN_LSTM_Deep',
    'Test accuracy': round(test_acc, 4),
    'F1 (macro)': round(f1_score(y_test, pred_main, average='macro'), 4),
    'F1 (weighted)': round(f1_score(y_test, pred_main, average='weighted'), 4),
}])
summary.to_csv('results/test_summary.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
epochs_range = range(1, len(history_main['val_acc']) + 1)
axes[0].plot(epochs_range, history_main['train_loss'], label='Train loss', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_title('Training loss', fontweight='bold'); axes[0].grid(alpha=0.3)
axes[0].legend()
ax2 = axes[0].twinx()
ax2.plot(epochs_range, history_main['val_acc'], color='green', label='Val accuracy (EMA)', linewidth=2)
ax2.legend(loc='center right')

cm = confusion_matrix(y_test, pred_main)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Norm', 'Supra', 'Ventr', 'Fusion', 'Unk'],
            yticklabels=['Norm', 'Supra', 'Ventr', 'Fusion', 'Unk'])
axes[1].set_title(f'Confusion Matrix - Test (acc {test_acc:.4f})', fontweight='bold')
axes[1].set_ylabel('True Label'); axes[1].set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('results/training_and_confusion.png', dpi=200)
plt.show()
print("Saved: results/training_and_confusion.png")

## 8. (Optional) Ensemble and test-time augmentation
Trains a second, different architecture (`ECG_ResNet_SE`) and compares five options on the validation split:
each model alone, the CNN-LSTM with test-time augmentation (TTA, averaging over small time stretches), and the
average of both models with and without TTA. The option with the best validation accuracy is reported on the test set.

The web application uses only the single CNN-LSTM model; deploying the ensemble would require backend changes.
Set `CONFIG['run_ensemble'] = False` to skip this section.

In [ ]:
@torch.no_grad()
def predict_proba_tta(model, X_raw, alphas=(0.97, 1.0, 1.03)):
    # Average predictions over small time-stretch variants of each (raw) beat
    probs = []
    for a in alphas:
        alpha = torch.full((len(X_raw), 1), a, device=device)
        probs.append(predict_proba(model, normalize(time_stretch(X_raw, alpha))))
    return np.mean(probs, axis=0)

if CONFIG['run_ensemble']:
    print("=" * 80)
    print("TRAINING ECG_ResNet_SE (ensemble member)")
    print("=" * 80)
    model_res, history_res = train_model(ECG_ResNet_SE(num_classes=5, dropout=CONFIG['dropout']),
                                         'models/resnet_se.pth', seed=SEED + 1)

    # Decide on the validation set which combination to trust, then report test
    candidates = {
        'CNN-LSTM': (predict_proba(model_main, X_val_t), probs_main),
        'CNN-LSTM + TTA': (predict_proba_tta(model_main, X_val_raw_t), predict_proba_tta(model_main, X_test_raw_t)),
        'ResNet-SE': (predict_proba(model_res, X_val_t), predict_proba(model_res, X_test_t)),
    }
    candidates['Ensemble'] = tuple((a + b) / 2 for a, b in zip(candidates['CNN-LSTM'], candidates['ResNet-SE']))
    candidates['Ensemble + TTA'] = tuple(
        (a + b) / 2 for a, b in zip(candidates['CNN-LSTM + TTA'],
                                    (predict_proba_tta(model_res, X_val_raw_t), predict_proba_tta(model_res, X_test_raw_t)))
    )

    rows = []
    for name, (pv, pt) in candidates.items():
        rows.append({'Model': name,
                     'Val acc': accuracy_score(y_val, pv.argmax(1)),
                     'Test acc': accuracy_score(y_test, pt.argmax(1)),
                     'Test F1 (macro)': f1_score(y_test, pt.argmax(1), average='macro')})
    ens_df = pd.DataFrame(rows).round(4)
    print(ens_df.to_string(index=False))
    ens_df.to_csv('results/ensemble_comparison.csv', index=False)

    best_name = ens_df.sort_values('Val acc', ascending=False).iloc[0]['Model']
    print(f"\nBest on validation: {best_name}")
    print(classification_report(y_test, candidates[best_name][1].argmax(1), target_names=class_names, digits=4))

## 9. Deployment check
Reloads the saved files the same way the web application's backend does (`backend/inference.py`): a fresh
`ECG_CNN_LSTM_Deep(num_classes=5, dropout=0.45)` on the CPU, `scaler.transform` on each beat, and a
`(N, 1, 187)` input tensor. The accuracy must match the evaluation above.

In [ ]:
backend_model = ECG_CNN_LSTM_Deep(num_classes=5, dropout=0.45)
backend_model.load_state_dict(torch.load('models/best_model.pth', map_location='cpu'))
backend_model.eval()
backend_scaler = joblib.load('models/scaler.pkl')

with torch.no_grad():
    X_backend = torch.from_numpy(backend_scaler.transform(X_test_raw)).float().unsqueeze(1)
    backend_pred = torch.cat([backend_model(X_backend[i:i + 2048]).argmax(1)
                              for i in range(0, len(X_backend), 2048)]).numpy()

backend_acc = accuracy_score(y_test, backend_pred)
print(f"Backend-style test accuracy: {backend_acc:.4f} (GPU eval: {test_acc:.4f})")
assert abs(backend_acc - test_acc) < 1e-3, "Mismatch between deployment-style and notebook evaluation"
print("models/best_model.pth and models/scaler.pkl are ready for the backend")

## 10. Download the trained files
Copy `best_model.pth` and `scaler.pkl` into the project's `models/` folder (the web application's default),
or point the `MODEL_DIR` environment variable at the folder that contains them.

In [ ]:
try:
    from google.colab import files
    print("Downloading model weights...")
    files.download('models/best_model.pth')
    print("Downloading scaler...")
    files.download('models/scaler.pkl')
except ImportError:
    print("Not running in Colab: the files are in ./models/")

## 11. Limitations
* **Intra-patient split.** The Kaggle training and test splits are divided by heartbeat, not by patient, so beats
  from the same person can appear in both. Results are therefore higher than they would be for new patients.
  An inter-patient evaluation (training and testing on different records) gives a more realistic estimate.
* **Single lead.** The model sees one ECG lead (MLII) and a single beat at a time. It has no rhythm context, such
  as RR intervals, which clinicians use to identify supraventricular beats.
* **Class imbalance.** The S and F classes have few examples, and their results vary more between training runs.
* **Not a medical device.** This work is for research and education only.

## References
1. G. B. Moody and R. G. Mark, "The impact of the MIT-BIH Arrhythmia Database," *IEEE Engineering in Medicine and
   Biology Magazine*, vol. 20, no. 3, pp. 45–50, 2001.
2. A. L. Goldberger et al., "PhysioBank, PhysioToolkit, and PhysioNet: Components of a new research resource for
   complex physiologic signals," *Circulation*, vol. 101, no. 23, pp. e215–e220, 2000.
3. M. Kachuee, S. Fazeli and M. Sarrafzadeh, "ECG heartbeat classification: A deep transferable representation,"
   in *Proc. IEEE International Conference on Healthcare Informatics (ICHI)*, 2018. Source of the pre-segmented
   Kaggle dataset `shayanfazeli/heartbeat`.
4. ANSI/AAMI EC57, *Testing and reporting performance results of cardiac rhythm and ST segment measurement
   algorithms* (heartbeat class grouping N, S, V, F, Q).
5. K. He, X. Zhang, S. Ren and J. Sun, "Deep residual learning for image recognition," in *Proc. CVPR*, 2016.
6. J. Hu, L. Shen and G. Sun, "Squeeze-and-excitation networks," in *Proc. CVPR*, 2018.
7. P. de Chazal, M. O'Dwyer and R. B. Reilly, "Automatic classification of heartbeats using ECG morphology and
   heartbeat interval features," *IEEE Transactions on Biomedical Engineering*, vol. 51, no. 7, pp. 1196–1206, 2004
   (inter-patient evaluation protocol).